In [1]:
%pylab inline
import gc
import fitsio
import catutil
import numpy as np
import numpy.lib.recfunctions as rfn

import astropy.table as astTable

Populating the interactive namespace from numpy and matplotlib


# Biases 
+ shear estimation multiplicative bias ($m$)
+ selection multiplicative bias ($m_{\rm sel}$)
+ selection additive bias ($a_{\rm sel}$)


In [2]:
nzs    =   4
dm1=[0.07872,0.09704,-0.01498]
# here I use the dnnz since the mock is based on dnnz
pzname  =   'dnnz' #demp/mizuki/dnnz
catFname=   'wrkDir/g3Sim/catalog_obs_reGaus_no_m/%s_no_m.fits' #%fieldname
biaFname=   'wrkDir/g3Sim/catalog_obs_reGaus_blinded_m/xiangchong-3blind/%s_blinded_%d_m.fits' #%(fieldname,bid)
pzsFname=   'wrkDir/catalog/%s_%s_zmask.fits' #%(fieldname,pzname)
outFname=   'wrkDir/g3Sim/catalog_obs_reGaus_smallcat_xiangchong/%s.fits' #%(fieldname)
# 4 bins
dataG  =   [[],[],[],[]]
for fieldname in catutil.field_names:
    if os.path.isfile(outFname%fieldname):
        tmp=fitsio.read(outFname%fieldname)
    else:
        tmp=[]
        for ib in range(3):
            bdata   =   fitsio.read(biaFname %(fieldname,ib))
            bdata=rfn.rename_fields(bdata,{'i_hsmshaperegauss_derived_shear_bias_m':'bias_m_b%d'%ib})
            bdata['bias_m_b%d'%ib]=bdata['bias_m_b%d'%ib]-dm1[ib]
            bdata=bdata[['bias_m_b%d'%ib,]]
            tmp.append(bdata)
        sdata   =   fitsio.read(catFname %(fieldname))[['i_hsmshaperegauss_derived_weight','i_hsmshaperegauss_derived_rms_e',\
                                                        'i_apertureflux_10_mag','i_hsmshaperegauss_resolution',\
                                                       'i_hsmshaperegauss_e1','i_hsmshaperegauss_derived_shear_bias_c1',\
                                                       'i_hsmshaperegauss_e2','i_hsmshaperegauss_derived_shear_bias_c2',\
                                                       'i_ra','i_dec']]
        tmp.append(sdata)
        tmp=rfn.merge_arrays(tmp, flatten=True, usemask=False)
        fitsio.write(outFname%fieldname,tmp)
    # selection using measured photo-z (not the true source photo-z)
    for iz in range(nzs):
        zmsk   =   fitsio.read(pzsFname %(fieldname,pzname))['%s_mask_%sbin' %(pzname,iz)]
        # merge observational fields
        dataG[iz].append(tmp[zmsk])
    del bdata,sdata,zmsk,tmp
    gc.collect()
dataG=[np.hstack(dataG[i]) for i in range(nzs)]

In [3]:
data=[]
names=[]
for ib in range(3):
    names.append('m_shear_cat%d' %ib)
names.extend(['response','m_sel','a_sel','m_sel_err','a_sel_err'])
print(names)
for iz in range(nzs):
    # in each redshfit bin
    wsum=np.sum(dataG[iz]['i_hsmshaperegauss_derived_weight'])
    zdat=[]
    for ib in range(3):
        m_shear=np.sum(dataG[iz]['bias_m_b%d' %ib]*dataG[iz]['i_hsmshaperegauss_derived_weight'])/wsum
        zdat.append(m_shear)
    response=   1.-np.sum(dataG[iz]['i_hsmshaperegauss_derived_rms_e']**2.*dataG[iz]['i_hsmshaperegauss_derived_weight'])/wsum
    zdat.append(response)
    row=tuple(zdat)+catutil.get_sel_bias(dataG[iz]['i_hsmshaperegauss_derived_weight'],dataG[iz]['i_apertureflux_10_mag'],dataG[iz]['i_hsmshaperegauss_resolution'])
    data.append(row)
    print(row)
tab=astTable.Table(rows=data,names=names)
tab.write('wrkDir/catalog/shear_biases_%s.csv'%pzname)

['m_shear_cat0', 'm_shear_cat1', 'm_shear_cat2', 'response', 'm_sel', 'a_sel', 'm_sel_err', 'a_sel_err']
(-0.09141873382555632, -0.04141986637719917, 0.008576928651350645, 0.8407182287610664, 0.009205343535801429, 0.0031953056659898005, 0.0006539147131223044, 0.000451824547471789)
(-0.10856406026726798, -0.05856519281891106, -0.008568397790361242, 0.8469750725856124, 0.01573986888212821, 0.006766819785054489, 0.0014204144687904687, 0.0009344313797495946)
(-0.11036946813340669, -0.06037060068504935, -0.010373805656499539, 0.8464418502264891, 0.015100718683580937, 0.008362438526165347, 0.0019340869826879213, 0.0011627008240484)
(-0.11541040947253108, -0.06541154202417392, -0.015414746995624005, 0.8410178590969258, 0.012108216412363197, 0.008442651379087425, 0.0021458840889916684, 0.0011996138863263287)
